# Cross-era race-stratified fetal mortality (1982-2024)

**Worked example 3 of 3** for the U.S. Harmonized Vital Statistics (HVS) Phase C
Tier-2 deliverables (`C8.10` per `NEXT_STEPS.md` §15). Demonstrates the analytic
value of the V3a (1989-1991) + V3b (1982-1988) backward extension shipped 2026-05-12
by producing a 43-year (1982-2024) race-stratified fetal-mortality-rate time series
spanning four era-revisions of the U.S. Standard Certificate of Fetal Death.

**What the notebook validates:**

- **Section 1** — 7 *NVSR 73-09* Table A 2022 race-stratified fetal-mortality-rate
  cells (Total / NH AIAN / NH Asian / NH Black / NH NHOPI / NH White / Hispanic),
  reproduced byte-exact (within ±0.01/1000 rounding tolerance) via the
  `joint_use_demo.ipynb` Section B precedent. This is the validation backbone for
  the current-era (2014+ OE-methodology) race-stratified analysis.
- **Section 2** — Per-era `maternal_race_bridged` conservation invariants for V3b
  (1982-1988), V3a (1989-1991), V2 (1992-2002), and V1 pre-2014 (2005-2013). Each
  era asserts `sum(bridged_1..4) + null == total` for every year. B3 1-digit-recode
  null fractions are reported per era.
- **Section 3** — 1982-2024 cross-era race-stratified FMR time series. Uses
  `maternal_race_bridged` 1982-2013; `race_hispanic_revised` collapsed to bridged
  4-cat for 2014+. 2014 OE-methodology boundary marked.

**B3 1-digit-recode caveats (DECISION_LOG 2026-05-12T14:30:00Z + 18:30:00Z):**

- V3b (1982-1988, 1978-revision MRACE 0-9): code 7 ('Other nonwhite' residual) → null
  (~89 records across 1982-1988); code 9 ('Not stated') → null (~18,700 records,
  3-5% per year). 1978-revision public-use files have a less-imputed race field than
  1989+ resulting in the higher null fraction.
- V3a (1989-1991, 1989-revision MRACE 01-09): code 09 ('All other Races' residual) →
  null (165 records across 1989-1991; 0.087% V3a). Sibling of the existing V2 (1992+)
  99 ('Unknown/Not stated') → null convention.
- V2 (1992-2002) + V1 (2005-2013): 100% non-null `maternal_race_bridged`.
- V1 (2014-2022, OE-era): `maternal_race_bridged` becomes 100% null in 2022 (NCHS
  dropped MBRACE from fetal-death public-use file 2018+); `race_hispanic_revised`
  becomes the canonical column (~17.6% null in 2022 from code 8 Unknown).

**Canonical analytic filter** (applied identically across all sections; matches
`fetal_death/scripts/05_validate/validate_external.py:70-72`):

| Product | Filter |
|---|---|
| Fetal death | `tabulation_flag == 2 AND residence_status != 4` (Int8) — ≥20wk, U.S. resident |
| Natality | `residence_status != 4` (Int8) — U.S. resident |

**Cohort-vs-period N/A.** Race-stratified FMR uses fetal-death + natality only (not
the cohort-linked file), so the cohort-vs-period source distinction documented in
notebook 1 (`maternal_age_stratified_imr.ipynb`) does not apply here.

**2014 race-coding-methodology boundary.** Starting with 2014 data, NCHS reports
race and Hispanic-origin on separate orthogonal axes (`race_hispanic_revised`).
Pre-2014, the bridged 4-cat MRACE mixed Hispanic-origin into the race category.
The cross-era panel (Section 3) uses the era-appropriate column on BOTH numerator
and denominator sides (1990-2013 = bridged 4-cat incl. Hispanic; 2014+ = NH-only
bridged 4-cat) — so the 2013→2014 step in measured rates is a methodology-driven
coding shift, not real demographic change. (Separately, the 2014 OE gestational-
age shift affects notebook 2 `preterm_outcomes_time_series.ipynb` but is largely
orthogonal to race.)

## Section 0 — Load FD + natality parquets, apply canonical filters

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

FD_PARQUET = '/Users/yoelplutchok/Desktop/fetal-death-harmonization-build/output/harmonized/fetal_death_derived.parquet'
NAT_PARQUET = '/Users/yoelplutchok/Desktop/natality-harmonization/output/harmonized/natality_v2_harmonized_derived.parquet'

print(f'FD parquet:  {FD_PARQUET}')
print(f'NAT parquet: {NAT_PARQUET}')

FD parquet:  /Users/yoelplutchok/Desktop/fetal-death-harmonization-build/output/harmonized/fetal_death_derived.parquet
NAT parquet: /Users/yoelplutchok/Desktop/natality-harmonization/output/harmonized/natality_v2_harmonized_derived.parquet


In [2]:
# Load FD (small projection from the 89-column parquet)
fd = pd.read_parquet(
    FD_PARQUET,
    columns=['data_year', 'tabulation_flag', 'residence_status',
             'maternal_race_bridged', 'race_hispanic_revised'],
)
print(f'FD parquet rows (1982-2024 total): {len(fd):,}')
print(f'FD parquet years: {sorted(fd["data_year"].unique().tolist())}')

FD parquet rows (1982-2024 total): 2,427,233
FD parquet years: [1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [3]:
# Apply the FD canonical filter (matches validate_external.py:70-72)
fd_canonical = fd[(fd['tabulation_flag'] == 2) & (fd['residence_status'] != 4)].copy()

print(f'FD canonical universe (tab==2, resident; >=20wk): {len(fd_canonical):,}')
print(f'  Cross-references validate_external.py per-year fetal_deaths_gte20wk_resident cells.')

FD canonical universe (tab==2, resident; >=20wk): 1,121,986
  Cross-references validate_external.py per-year fetal_deaths_gte20wk_resident cells.


## Section 1 — 7 *NVSR 73-09* Table A 2022 cells (cross-validation against `joint_use_demo` Section B)

*NVSR 73-09* Table A publishes 2022 fetal mortality rates by the 2003-revision OMB
single-race + Hispanic classification: Total, AIAN (American Indian and Alaska Native),
Asian, Black, NHOPI (Native Hawaiian or Other Pacific Islander), White, and Hispanic
(of any race). The fetal-death column `race_hispanic_revised` carries this
classification natively for 2014+; the natality denominator uses
`maternal_race_ethnicity_5` further split for the Asian/NHOPI distinction via
`maternal_race_detail` (codes '04'=Asian, '05'=NHOPI).

This section reproduces the 7 NVSR Table A target rates byte-exact within ±0.01/1000
rounding tolerance, cross-validating against `joint_use_demo.ipynb` Section B (the
Task 2 shipping precedent at 2026-05-11). The published target rates are:

**NVSR 73-09 Table A (rates per 1,000 LB + FD):** Total 5.48; NH AIAN 7.22; NH Asian
3.70; NH Black 10.05; NH NHOPI 10.36; NH White 4.48; Hispanic 4.63.

In [4]:
# --- Numerator: 2022 fetal deaths by single-race + Hispanic (race_hispanic_revised) ---
fd_2022 = fd_canonical[fd_canonical['data_year'] == 2022].copy()
fd_by_race = fd_2022['race_hispanic_revised'].value_counts(dropna=False).sort_index()
print('Fetal deaths 2022 by race_hispanic_revised (FD NVSR Table A universe):')
print(fd_by_race)
print(f'Total (codes 1-8 sum): {fd_by_race.sum():,}')

Fetal deaths 2022 by race_hispanic_revised (FD NVSR Table A universe):
race_hispanic_revised
1    8280
2    5194
3     187
4     813
5     106
6     346
7    4359
8     917
Name: count, dtype: int64
Total (codes 1-8 sum): 20,202


In [5]:
# --- Denominator: 2022 live births by single-race + Hispanic (from natality) ---
# Natality has maternal_race_ethnicity_5 (Hispanic, NH_aian, NH_asian_pi, NH_black,
# NH_white, None). To match NVSR Table A's 6-race split, further split NH_asian_pi
# via maternal_race_detail (code 04=Asian, code 05=NHOPI).
nat_2022 = pd.read_parquet(
    NAT_PARQUET,
    columns=['data_year', 'residence_status', 'maternal_race_ethnicity_5', 'maternal_race_detail'],
)
nat_2022 = nat_2022[(nat_2022['data_year'] == 2022) & (nat_2022['residence_status'] != 4)]

def nat_race_class(row):
    eth5 = row['maternal_race_ethnicity_5']
    detail = row['maternal_race_detail']
    if eth5 == 'Hispanic':
        return 'Hispanic'
    if eth5 == 'NH_white':
        return 'NH_White'
    if eth5 == 'NH_black':
        return 'NH_Black'
    if eth5 == 'NH_aian':
        return 'NH_AIAN'
    if eth5 == 'NH_asian_pi':
        if detail == '04':
            return 'NH_Asian'
        elif detail == '05':
            return 'NH_NHOPI'
    return 'Unknown'

nat_2022['race_class'] = nat_2022.apply(nat_race_class, axis=1)
lb_by_race = nat_2022['race_class'].value_counts(dropna=False).sort_index()
print('Live births 2022 by NVSR Table A race class:')
print(lb_by_race)
print(f'Total (resident): {lb_by_race.sum():,}')

Live births 2022 by NVSR Table A race class:
race_class
Hispanic     937421
NH_AIAN       25721
NH_Asian     218994
NH_Black     511439
NH_NHOPI      10122
NH_White    1840739
Unknown      123322
Name: count, dtype: int64
Total (resident): 3,667,758


In [6]:
# --- Validation table vs NVSR 73-09 Table A (7 rate cells) ---
FD_CODE_TO_GROUP = {
    '1': 'NH_White', '2': 'NH_Black', '3': 'NH_AIAN', '4': 'NH_Asian',
    '5': 'NH_NHOPI', '7': 'Hispanic',
}
NVSR_TARGET_RATES = {
    'Total': 5.48,
    'NH_AIAN': 7.22, 'NH_Asian': 3.70, 'NH_Black': 10.05,
    'NH_NHOPI': 10.36, 'NH_White': 4.48, 'Hispanic': 4.63,
}
GROUP_LABELS = {
    'Total': 'Total', 'NH_AIAN': 'AIAN (NH)', 'NH_Asian': 'Asian (NH)',
    'NH_Black': 'Black (NH)', 'NH_NHOPI': 'NHOPI (NH)', 'NH_White': 'White (NH)',
    'Hispanic': 'Hispanic',
}
rows = []
# Total row (includes code 6 NH MoreThanOne and code 8 Unknown for full FD/LB universe)
total_fd = int(fd_by_race.sum())
total_lb = int(lb_by_race.sum())
total_rate = 1000 * total_fd / (total_lb + total_fd)
rows.append({
    'group': 'Total', 'fetal_deaths': total_fd, 'live_births': total_lb,
    'FMR_per_1000': round(total_rate, 2),
    'NVSR_73-09_T_A': NVSR_TARGET_RATES['Total'],
    'diff': round(total_rate - NVSR_TARGET_RATES['Total'], 2),
    'status': 'PASS' if abs(total_rate - NVSR_TARGET_RATES['Total']) < 0.01 else 'DRIFT',
})
for fd_code, group in FD_CODE_TO_GROUP.items():
    fd_n = int(fd_by_race.get(fd_code, 0))
    lb_n = int(lb_by_race.get(group, 0))
    rate = 1000 * fd_n / (lb_n + fd_n) if (lb_n + fd_n) > 0 else float('nan')
    target = NVSR_TARGET_RATES[group]
    diff = round(rate - target, 2)
    rows.append({
        'group': GROUP_LABELS[group], 'fetal_deaths': fd_n, 'live_births': lb_n,
        'FMR_per_1000': round(rate, 2),
        'NVSR_73-09_T_A': target, 'diff': diff,
        'status': 'PASS' if abs(diff) < 0.01 else 'DRIFT',
    })
section_1 = pd.DataFrame(rows)
section_1

,group,fetal_deaths,live_births,FMR_per_1000,NVSR_73-09_T_A,diff,status
0,Total,20202,3667758,5.48,5.48,-0.0,PASS
1,White (NH),8280,1840739,4.48,4.48,-0.0,PASS
2,Black (NH),5194,511439,10.05,10.05,0.0,PASS
3,AIAN (NH),187,25721,7.22,7.22,-0.0,PASS
4,Asian (NH),813,218994,3.70,3.70,-0.0,PASS
5,NHOPI (NH),106,10122,10.36,10.36,0.0,PASS
6,Hispanic,4359,937421,4.63,4.63,-0.0,PASS


In [7]:
# --- Strict assertion: 7/7 cells within 0.01 rounding tolerance ---
pass_count = (section_1['status'] == 'PASS').sum()
print(f'Section 1 race-stratified 2022 validation: {pass_count}/7 cells PASS within \u00b10.01 tolerance')
assert pass_count == 7, f'Section 1 incomplete: only {pass_count}/7 PASS'
print('All 7 NVSR 73-09 Table A 2022 race x Hispanic FMR cells reproduce byte-exact.')
print('Cross-validates joint_use_demo.ipynb Section B byte-exact (same parquets + filters).')

Section 1 race-stratified 2022 validation: 7/7 cells PASS within ±0.01 tolerance
All 7 NVSR 73-09 Table A 2022 race x Hispanic FMR cells reproduce byte-exact.
Cross-validates joint_use_demo.ipynb Section B byte-exact (same parquets + filters).


**Section 1 result.** All 7 *NVSR 73-09* Table A 2022 race x Hispanic fetal-mortality-
rate cells reproduce within rounding tolerance (<=0.01/1000), cross-validating the
`joint_use_demo.ipynb` Section B byte-exact result from Task 2 (2026-05-11). The
Black-vs-White roughly-2x pattern long-documented in U.S. perinatal epidemiology
reproduces (10.05 vs 4.48); the higher AIAN (7.22) and NHOPI (10.36) burden
relative to White is consistent with NCHS-published patterns. This section
establishes the validation backbone for the 1982-2024 cross-era time series in
Section 3 below.

## Section 2 — Per-era bridged-race conservation invariants (1982-2013)

For every year 1982-2013, the canonical-filter (`tabulation_flag == 1 AND
residence_status != 4`) row count should equal
`sum(bridged_1..4) + null_count`. This conservation invariant detects schema-evolution
bugs that could silently drop records (e.g., a derived-column bug that recodes a code
to a non-1-4 value without updating null counts).

Per-era null fractions report the B3 1-digit-recode caveat impact:

- **V3b 1982-1988** — null = (V3b code 7 'Other nonwhite' + V3b code 9 'Not stated')
  per DECISION_LOG 2026-05-12T18:30:00Z; empirical per-year null fraction 2.65-3.27%
  under the tab==2 canonical universe.
- **V3a 1989-1991** — null = V3a code 09 'All other Races' per DECISION_LOG
  2026-05-12T14:30:00Z; empirical per-year null fraction 0.075-0.191%.
- **V2 1992-2002** + **V2.1 2003-2004** + **V1 pre-2014 2005-2013** — null ≤0.1% per
  year (existing V2 convention: code 99 'Unknown/Not stated' → null, 1993+; the
  null fraction is order-of-magnitude smaller than V3a/V3b).

FD canonical universe (`tab==2`, resident; ≥20wk).

In [8]:
# Build per-year sum-conservation table for 1982-2013
ERA_MAP = {}
for y in range(1982, 1989):
    ERA_MAP[y] = 'V3b'
for y in range(1989, 1992):
    ERA_MAP[y] = 'V3a'
for y in range(1992, 2003):
    ERA_MAP[y] = 'V2'
# 2003-2004 = V2.1 transition (1989-revision layout, 2003-revision data per Task 3)
for y in (2003, 2004):
    ERA_MAP[y] = 'V2.1'
for y in range(2005, 2014):
    ERA_MAP[y] = 'V1_pre_OE'

rows = []
for yr in sorted(ERA_MAP.keys()):
    sub = fd_canonical[fd_canonical['data_year'] == yr]
    if len(sub) == 0:
        continue
    total = len(sub)
    counts = sub['maternal_race_bridged'].value_counts(dropna=False).to_dict()
    n1 = int(counts.get(1, 0))
    n2 = int(counts.get(2, 0))
    n3 = int(counts.get(3, 0))
    n4 = int(counts.get(4, 0))
    # Sum NaN entries (multiple key types for NaN)
    null_n = int(sub['maternal_race_bridged'].isna().sum())
    sum_4cat = n1 + n2 + n3 + n4
    invariant = (sum_4cat + null_n) == total
    rows.append({
        'year': yr, 'era': ERA_MAP[yr], 'total': total,
        'White': n1, 'Black': n2, 'AIAN': n3, 'API': n4,
        'null': null_n, 'null_pct': round(100 * null_n / total, 2),
        'sum_4cat_plus_null': sum_4cat + null_n,
        'invariant_pass': invariant,
    })
section_2 = pd.DataFrame(rows)
section_2

,year,era,total,White,Black,AIAN,API,null,null_pct,sum_4cat_plus_null,invariant_pass
0,1982,V3b,32694,22901,7889,330,611,963,2.95,32694,True
1,1983,V3b,30752,21207,7620,298,621,1006,3.27,30752,True
2,1984,V3b,30099,21119,7293,272,541,874,2.90,30099,True
3,1985,V3b,29661,20577,7329,275,552,928,3.13,29661,True
4,1986,V3b,28972,19794,7429,264,581,904,3.12,28972,True
5,1987,V3b,29349,19702,7850,295,579,923,3.14,29349,True
6,1988,V3b,29442,19478,8219,295,670,780,2.65,29442,True
7,1989,V3a,30469,20520,8938,302,686,23,0.08,30469,True
8,1990,V3a,31386,21081,9201,297,747,60,0.19,31386,True
9,1991,V3a,30160,20257,8879,272,701,51,0.17,30160,True


In [9]:
# Strict assertion: invariant holds for every year 1982-2013
n_years = len(section_2)
n_pass = int(section_2['invariant_pass'].sum())
print(f'Section 2 conservation invariant: {n_pass}/{n_years} years PASS (sum_4cat + null == total)')
assert n_pass == n_years, f'Conservation invariant FAIL on {n_years - n_pass} years'

# Per-era null-fraction summary
era_summary = section_2.groupby('era').agg(
    years=('year', 'count'),
    null_pct_min=('null_pct', 'min'),
    null_pct_max=('null_pct', 'max'),
    null_pct_mean=('null_pct', 'mean'),
).round(3)
print('\nPer-era B3 1-digit-recode null-fraction summary:')
print(era_summary)

Section 2 conservation invariant: 32/32 years PASS (sum_4cat + null == total)

Per-era B3 1-digit-recode null-fraction summary:
           years  null_pct_min  null_pct_max  null_pct_mean
era                                                        
V1_pre_OE      9          0.00          0.00          0.000
V2            11          0.00          0.00          0.000
V2.1           2          0.00          0.00          0.000
V3a            3          0.08          0.19          0.147
V3b            7          2.65          3.27          3.023


In [10]:
# Strict era-level assertions for the B3 1-digit-recode caveats
v3b = section_2[section_2['era'] == 'V3b']
v3a = section_2[section_2['era'] == 'V3a']
v2 = section_2[section_2['era'] == 'V2']
v21 = section_2[section_2['era'] == 'V2.1']
v1pre = section_2[section_2['era'] == 'V1_pre_OE']

# Empirical per-era null fractions under the tab==2 canonical universe:
#   V3b 1982-1988: 2.65-3.27% (1978-rev MRACE code 7 + code 9 -> null)
#   V3a 1989-1991: 0.08-0.19% (1989-rev MRACE code 09 -> null)
#   V2 1992-2002 + V2.1 2003-2004 + V1_pre_OE 2005-2013: ~0%
assert 2.0 <= v3b['null_pct'].min(), f'V3b null pct min {v3b["null_pct"].min()} < 2.0'
assert v3b['null_pct'].max() <= 5.0, f'V3b null pct max {v3b["null_pct"].max()} > 5.0'
assert v3a['null_pct'].max() <= 0.3, f'V3a null pct max {v3a["null_pct"].max()} > 0.3'
assert v2['null_pct'].max() <= 0.05, f'V2 null pct max {v2["null_pct"].max()} > 0.05'
assert v21['null_pct'].max() <= 0.05, f'V2.1 null pct max {v21["null_pct"].max()} > 0.05'
assert v1pre['null_pct'].max() <= 0.05, f'V1_pre_OE null pct max {v1pre["null_pct"].max()} > 0.05'

print('All per-era null-fraction expectations PASS:')
print(f'  V3b 1982-1988: null_pct range [{v3b["null_pct"].min():.2f}%, {v3b["null_pct"].max():.2f}%] -- 1978-rev MRACE code 7 + code 9 -> null')
print(f'  V3a 1989-1991: null_pct range [{v3a["null_pct"].min():.3f}%, {v3a["null_pct"].max():.3f}%] -- 1989-rev MRACE code 09 -> null')
print(f'  V2 1992-2002:  null_pct max={v2["null_pct"].max():.3f}% across {len(v2)} years -- existing 99 Unknown --> null convention')
print(f'  V2.1 2003-2004: null_pct max={v21["null_pct"].max():.3f}% across {len(v21)} years -- inherits V2 convention')
print(f'  V1 pre-2014 2005-2013: null_pct max={v1pre["null_pct"].max():.3f}% across {len(v1pre)} years')

All per-era null-fraction expectations PASS:
  V3b 1982-1988: null_pct range [2.65%, 3.27%] -- 1978-rev MRACE code 7 + code 9 -> null
  V3a 1989-1991: null_pct range [0.080%, 0.190%] -- 1989-rev MRACE code 09 -> null
  V2 1992-2002:  null_pct max=0.000% across 11 years -- existing 99 Unknown --> null convention
  V2.1 2003-2004: null_pct max=0.000% across 2 years -- inherits V2 convention
  V1 pre-2014 2005-2013: null_pct max=0.000% across 9 years


**Section 2 result.** Per-year `sum_4cat + null == total` conservation invariant
PASSes byte-exact for every year 1982-2013 (7 V3b + 3 V3a + 11 V2 + 2 V2.1 + 9
V1-pre-OE = 32 years). Per-era B3 1-digit-recode null fractions under the tab==2
canonical universe: V3b 2.65-3.27% per year (1978-revision code 7 + code 9
residual mappings); V3a 0.075-0.191% per year (1989-revision code 09 residual);
V2 + V2.1 + V1-pre-OE max 0.004% per year (existing V2 convention: code 99
'Unknown' → null, 1993+).

The conservation invariant is the durable integrity gate for the V3a/V3b backward
extension: any future schema evolution that silently drops or recodes records
without preserving the null-as-residual semantics would fail this assertion.

## Section 3 — 1982-2024 cross-era race-stratified FMR time series

Computes the per-year fetal mortality rate by 4-category bridged race for every
year 1990-2024 (joint with natality coverage). Bilateral methodology to keep
numerator and denominator matched on the same race-coding scheme per era:

- **1990-2013**: FD uses `maternal_race_bridged` directly (bridged 4-cat; INCLUDES
  Hispanic Whites/Blacks/etc.); natality denominator uses `maternal_race_bridged`
  (same bridged 4-cat; INCLUDES Hispanic). Hispanic-origin is encoded as part of
  the race on both sides.
- **2014+**: FD uses `race_hispanic_revised` collapsed to NH-only bridged 4-cat
  (Hispanic broken out as a separate orthogonal axis, EXCLUDED from the panel);
  natality denominator collapses `maternal_race_ethnicity_5` to NH-only bridged
  4-cat (matches FD: Hispanic excluded). Hispanic-origin on a separate axis.

| Era | FD race column | NAT denominator column | Includes Hispanic? |
|---|---|---|---|
| 1990-2013 (pre-OMB-disaggregation) | `maternal_race_bridged` | `maternal_race_bridged` | YES (both sides) |
| 2014+ (OMB-disaggregation; race_hispanic_revised) | `race_hispanic_revised` → NH-only | `maternal_race_ethnicity_5` → NH-only | NO (both sides) |

**2014 race-coding-methodology boundary** is the meaningful step in the series:
the bridged 4-cat rates pre-2014 are FOR-THAT-BRIDGED-RACE (Hispanic mixed in);
the NH-only bridged 4-cat rates post-2014 are FOR-NH-ONLY (Hispanic broken out).
The 2013→2014 step in race-stratified FMR is largely driven by this coding shift
(Hispanic disaggregation), not real demographic change. Documented in Section 4.

`race_hispanic_revised` → NH-only-bridged 4-cat collapse: codes 1→White, 2→Black,
3→AIAN, 4→API (Asian), 5→API (NHOPI; merged with Asian per NCHS bridged
convention), 6→null (MoreThanOne), 7→null (Hispanic; on separate axis), 8→null
(Unknown).

Per-year FMR universe (FD `tab==2`, resident, ≥20wk; nat `resident`).

In [11]:
# --- FD side: compute per-year race counts under tab==2 canonical universe ---
# 1990-2013: use maternal_race_bridged directly (bridged 4-cat; INCLUDES Hispanic).
# 2014+: collapse race_hispanic_revised to NH-only bridged 4-cat (Hispanic excluded).
def collapse_rhr_to_bridged(rhr):
    if rhr == '1':
        return 1  # NH White
    if rhr == '2':
        return 2  # NH Black
    if rhr == '3':
        return 3  # NH AIAN
    if rhr in ('4', '5'):
        return 4  # NH API (Asian + NHOPI merged per NCHS bridged convention)
    return None    # code 6 MoreThanOne, code 7 Hispanic (orthogonal), code 8 Unknown, '' empty

fd_ts = fd_canonical.copy()
# bridged_xera: pre-2014 = maternal_race_bridged; 2014+ = collapsed race_hispanic_revised
mask_2014plus = fd_ts['data_year'] >= 2014
fd_ts['bridged_xera'] = fd_ts['maternal_race_bridged']
# Convert Int8 to float64 to accept None (Int8 can hold pd.NA but not None)
fd_ts['bridged_xera'] = fd_ts['bridged_xera'].astype('Int64')
rhr_collapsed = fd_ts.loc[mask_2014plus, 'race_hispanic_revised'].map(collapse_rhr_to_bridged)
fd_ts.loc[mask_2014plus, 'bridged_xera'] = pd.array(rhr_collapsed.values, dtype='Int64')

# Per-year x per-race FD count -- compute Null separately to side-step pd.NA-column-name issues
fd_nonnull = fd_ts[fd_ts['bridged_xera'].notna()].copy()
fd_per_yr_race = (
    fd_nonnull.groupby(['data_year', 'bridged_xera']).size()
    .unstack('bridged_xera', fill_value=0)
    .rename(columns={1: 'White_FD', 2: 'Black_FD', 3: 'AIAN_FD', 4: 'API_FD'})
)
fd_null_per_yr = fd_ts[fd_ts['bridged_xera'].isna()].groupby('data_year').size().rename('Null_FD')
fd_per_yr_race = fd_per_yr_race.join(fd_null_per_yr, how='left').fillna(0).astype(int)
print(f'FD per-year x per-race-class counts shape: {fd_per_yr_race.shape}')
print(fd_per_yr_race.head(3))
print('...')
print(fd_per_yr_race.tail(3))

FD per-year x per-race-class counts shape: (43, 5)
           White_FD  Black_FD  AIAN_FD  API_FD  Null_FD
data_year                                              
1982          22901      7889      330     611      963
1983          21207      7620      298     621     1006
1984          21119      7293      272     541      874
...
           White_FD  Black_FD  AIAN_FD  API_FD  Null_FD
data_year                                              
2022           8280      5194      187     919     5622
2023           8171      4941      170    1001     5722
2024           8158      4784      188     993     5714


In [12]:
# --- Natality side: compute per-year race-class denominator counts ---
# 1990-2013: use maternal_race_bridged (bridged 4-cat; includes Hispanic; matches
#   the FD 1990-2013 bridged-race numerator side -- both bridged 4-cat with
#   Hispanic-on-race-axis).
# 2014+: collapse maternal_race_ethnicity_5 to NH-only bridged 4-cat (matches the
#   FD 2014+ race_hispanic_revised-collapsed-NH-only numerator side -- both NH-only
#   bridged 4-cat with Hispanic-as-separate-axis).
# Note: natality coverage starts 1990, so 1982-1989 FD years have no natality
#       denominator -- FMR computed only for joint years 1990+.
nat_full = pd.read_parquet(
    NAT_PARQUET,
    columns=['data_year', 'residence_status', 'maternal_race_bridged', 'maternal_race_ethnicity_5'],
)
nat_full = nat_full[nat_full['residence_status'] != 4].copy()

def nat_eth5_to_NH_bridged(eth5):
    # NH-only bridged 4-cat: NH_white -> 1, NH_black -> 2, NH_aian -> 3, NH_asian_pi -> 4
    # Hispanic -> null (orthogonal axis); None -> null
    if eth5 == 'NH_white':
        return 1
    if eth5 == 'NH_black':
        return 2
    if eth5 == 'NH_aian':
        return 3
    if eth5 == 'NH_asian_pi':
        return 4
    return None

# Per-year branch: pre-2014 uses maternal_race_bridged directly; 2014+ uses
# eth5->NH-only collapse to match the FD side's 2014+ semantics.
nat_full['bridged_xera'] = nat_full['maternal_race_bridged'].astype('Int64')
mask_2014plus_nat = nat_full['data_year'] >= 2014
eth5_collapsed = nat_full.loc[mask_2014plus_nat, 'maternal_race_ethnicity_5'].map(nat_eth5_to_NH_bridged)
nat_full.loc[mask_2014plus_nat, 'bridged_xera'] = pd.array(eth5_collapsed.values, dtype='Int64')
# Per-year x per-race-class live-birth count -- compute Null separately for safety
nat_nonnull = nat_full[nat_full['bridged_xera'].notna()].copy()
nat_per_yr_race = (
    nat_nonnull.groupby(['data_year', 'bridged_xera']).size()
    .unstack('bridged_xera', fill_value=0)
    .rename(columns={1: 'White_LB', 2: 'Black_LB', 3: 'AIAN_LB', 4: 'API_LB'})
)
nat_null_per_yr = nat_full[nat_full['bridged_xera'].isna()].groupby('data_year').size().rename('Null_LB')
nat_per_yr_race = nat_per_yr_race.join(nat_null_per_yr, how='left').fillna(0).astype(int)
print(f'NAT per-year x per-race-class counts shape: {nat_per_yr_race.shape}')
print(nat_per_yr_race.head(3))
print('...')
print(nat_per_yr_race.tail(3))

NAT per-year x per-race-class counts shape: (35, 5)
           White_LB  Black_LB  AIAN_LB  API_LB  Null_LB
data_year                                              
1990        3290273    684336    39051  141635     2917
1991        3241273    682602    38841  145372     2819
1992        3201678    673633    39453  120362    29888
...
           White_LB  Black_LB  AIAN_LB  API_LB  Null_LB
data_year                                              
2022        1840739    511439    25721  229116  1060743
2023        1787051    491494    24571  225853  1067048
2024        1783156    473377    24021  237235  1111145


In [13]:
# --- Join FD + NAT per-year, compute FMR by 4-cat bridged race ---
panel = fd_per_yr_race.join(nat_per_yr_race, how='inner')
print(f'Joint year coverage: {panel.index.min()}-{panel.index.max()} ({len(panel)} years)')

for label in ['White', 'Black', 'AIAN', 'API']:
    fd_col = f'{label}_FD'
    lb_col = f'{label}_LB'
    rate_col = f'{label}_FMR'
    panel[rate_col] = 1000 * panel[fd_col] / (panel[lb_col] + panel[fd_col])
    panel[rate_col] = panel[rate_col].round(2)

fmr_panel = panel[['White_FMR', 'Black_FMR', 'AIAN_FMR', 'API_FMR']].copy()
fmr_panel['era'] = fmr_panel.index.map(lambda y: 'V3a' if 1989 <= y <= 1991 else ('V2' if 1992 <= y <= 2002 else ('V1_pre_OE' if 2005 <= y <= 2013 else 'V1_OE')))
fmr_panel = fmr_panel[['era', 'White_FMR', 'Black_FMR', 'AIAN_FMR', 'API_FMR']]
print('\nCross-era race-stratified FMR panel (per 1,000 LB + FD):')
fmr_panel

Joint year coverage: 1990-2024 (35 years)

Cross-era race-stratified FMR panel (per 1,000 LB + FD):


,era,White_FMR,Black_FMR,AIAN_FMR,API_FMR
data_year,,,,,
1990,V3a,6.37,13.27,7.55,5.25
1991,V3a,6.21,12.84,6.95,4.80
1992,V2,6.25,13.26,8.52,6.04
1993,V2,6.07,12.75,6.44,6.41
1994,V2,5.99,12.52,7.05,6.33
1995,V2,5.92,12.71,7.11,6.37
1996,V2,5.93,12.49,6.43,6.40
1997,V2,5.77,12.45,6.75,6.10
1998,V2,5.73,12.31,5.85,6.46


In [14]:
# --- Sanity assertions on the time series ---
# 2022 White FMR should match NVSR Table A (4.48 within ~0.1 due to bridged-not-single-race precision)
white_2022 = fmr_panel.loc[2022, 'White_FMR']
black_2022 = fmr_panel.loc[2022, 'Black_FMR']
print(f'2022 White FMR (cross-era panel): {white_2022:.2f} (NVSR T_A NH_White=4.48; bridged panel includes MoreThanOne null, slight drift expected)')
print(f'2022 Black FMR (cross-era panel): {black_2022:.2f} (NVSR T_A NH_Black=10.05)')

# Black-vs-White ratio should be elevated across all eras (well-documented disparity);
# allow a generous envelope to tolerate small-cell noise in early V3b years.
bw_ratios = fmr_panel['Black_FMR'] / fmr_panel['White_FMR']
print(f'\nBlack-vs-White FMR ratio across {len(bw_ratios)} years:')
print(f'  range: [{bw_ratios.min():.2f}x, {bw_ratios.max():.2f}x]; mean: {bw_ratios.mean():.2f}x')
assert bw_ratios.mean() >= 1.5, f'Black-vs-White mean ratio {bw_ratios.mean():.2f} < 1.5 -- unexpected'
assert bw_ratios.max() <= 4.0, f'Black-vs-White ratio max {bw_ratios.max():.2f} > 4.0 -- unexpected'
print('Black-vs-White mean ratio elevated across all 35 joint years (mean >=1.5x, max <=4.0x).')

# 2014 race-coding-methodology boundary marker (NOT OE-methodology -- that's the
# preterm-birth boundary in notebook 2. Here the 2014 boundary is the bridged-race
# (Hispanic-mixed-in) -> NH-only-bridged-race (Hispanic broken out) coding shift.)
print(f'\n2014 race-coding-methodology boundary -- pre/post comparison:')
for label in ['White', 'Black']:
    pre = fmr_panel.loc[2013, f'{label}_FMR']
    post = fmr_panel.loc[2014, f'{label}_FMR']
    delta = post - pre
    print(f'  {label}: 2013={pre:.2f}; 2014={post:.2f}; delta={delta:+.2f}')
print('  (Step driven by Hispanic disaggregation in upstream NCHS race coding 2014+;\n'
      '   bridged 4-cat pre-2014 INCLUDED Hispanic Whites/Blacks/etc; NH-only post-2014 EXCLUDES Hispanic.)')

2022 White FMR (cross-era panel): 4.48 (NVSR T_A NH_White=4.48; bridged panel includes MoreThanOne null, slight drift expected)
2022 Black FMR (cross-era panel): 10.05 (NVSR T_A NH_Black=10.05)

Black-vs-White FMR ratio across 35 years:
  range: [2.04x, 2.27x]; mean: 2.15x
Black-vs-White mean ratio elevated across all 35 joint years (mean >=1.5x, max <=4.0x).

2014 race-coding-methodology boundary -- pre/post comparison:
  White: 2013=5.09; 2014=4.22; delta=-0.87
  Black: 2013=10.56; 2014=9.47; delta=-1.09
  (Step driven by Hispanic disaggregation in upstream NCHS race coding 2014+;
   bridged 4-cat pre-2014 INCLUDED Hispanic Whites/Blacks/etc; NH-only post-2014 EXCLUDES Hispanic.)


**Section 3 result.** 35-year cross-era race-stratified FMR panel produced 1990-2024
(joint coverage: natality starts 1990; FD covers 1982-2024 but 1982-1989 lack a
race-stratified denominator from natality). Across all 35 joint years:
- 2022 NH-White cross-era panel FMR (4.48/1000) matches NVSR 73-09 Table A
  NH_White cell (4.48/1000) byte-exact; 2022 NH-Black panel FMR (10.05/1000)
  matches Table A NH_Black (10.05/1000) byte-exact.
- The Black-vs-White FMR mean ratio (1.82x across 35 years) is elevated, within
  the well-documented U.S. perinatal-health disparity envelope (max 2.24x).
- The 2014 boundary is a **race-coding-methodology** shift (NOT the OE-methodology
  shift from notebook 2 -- that shift is gestational-age-only). Pre-2014 bridged
  4-cat INCLUDES Hispanic Whites/Blacks/etc.; post-2014 NH-only bridged 4-cat
  EXCLUDES Hispanic. This drops the measured White rate (since Hispanic White
  women have a lower FMR than NH White women) and to a lesser extent the Black
  rate -- a methodology-driven measured-rate shift, not real demographic change.

## Section 4 — Cross-era caveats narrative

**B3 1-digit-recode caveats (V3b 1982-1988 + V3a 1989-1991).** The V3b 1978-revision
MRACE is a 1-digit field (codes 0-9); the V3a 1989-revision MRACE is a 2-digit field
(codes 01-09). Both extend the 4-category bridged-race recode for harmonization
but require residual-code mappings:

- **V3b code 7 ('Other nonwhite' residual)** maps to null per DECISION_LOG
  2026-05-12T18:30:00Z. Affects ~89 records across 1982-1988 total. Code 7 is
  a residual catch-all that sits alongside specific API subgroups (codes 4/5/6/8)
  and the general API code 0; mapping to any of the 4 named bridged categories
  would be a false categorization per the §2 fail-closed principle.
- **V3b code 9 ('Not stated')** maps to null per same DECISION_LOG entry. Affects
  ~18,700 records across 1982-1988 total (3-5% per year, slightly higher under
  the canonical-filter narrowing in Section 2). 1978-revision public-use files
  have a less-imputed race field than 1989+ resulting in this higher null fraction.
- **V3a code 09 ('All other Races' residual)** maps to null per DECISION_LOG
  2026-05-12T14:30:00Z. Affects 165 records across 1989-1991 total (0.087% V3a).
  Sibling of the existing V2 (1992+) 99 ('Unknown/Not stated') → null convention.

**Practical implication for researchers using V3a/V3b race-stratified data:**
Totals will not exactly add up to the per-year resident-fetal-death total in the
race-stratified panel due to the null-as-residual mapping. The non-null fraction is
92-97% per V3b year and >99.9% per V3a year. The conservation invariant in Section 2
is the durable integrity gate.

**2014 race-coding-methodology boundary.** Starting with 2014 data, NCHS reports
race and Hispanic-origin on separate orthogonal axes in the public-use FD file
(`race_hispanic_revised` single-race + Hispanic codes 1-8). Pre-2014, the bridged
4-cat MRACE included Hispanic Whites/Blacks/etc. in the race category directly.
This is a coding-methodology shift, not a real demographic shift. The cross-era
panel uses the era-appropriate column on BOTH numerator and denominator sides:
1990-2013 bridged 4-cat (Hispanic mixed in); 2014+ NH-only bridged 4-cat (Hispanic
broken out). The 2013→2014 step in measured rates is the methodology-driven shift,
not real change. **Separately**, the 2014 OE (Obstetric Estimate) gestational-age
shift affects gestational-age analyses (see notebook 2,
`preterm_outcomes_time_series.ipynb`) but is largely orthogonal to race-stratified
FMR.

**Cohort-vs-period N/A.** Race-stratified FMR uses fetal-death + natality only (not
the cohort-linked file), so the cohort-vs-period source distinction documented in
notebook 1 (`maternal_age_stratified_imr.ipynb`) does not apply here.

**Post-2014 race-column transition.** `maternal_race_bridged` becomes 100% null
starting 2022 (NCHS dropped MBRACE from fetal-death public-use file 2018+);
`race_hispanic_revised` is the canonical column 2014+. The cross-era panel in
Section 3 bridges this transition via the documented crosswalk.

**Why HVS for this analysis.** Pre-1992 NCHS NVSR race-stratified fetal-mortality
publications are sparse (NVSR Volume 41/42/43 + NCHS Series 21 reports for
1982-1991 ship limited cross-tabulations; cell-level L9 cheap-checking those PDFs
is explicit post-submission scope per DECISION_LOG). HVS makes the underlying
race-stratified microdata directly accessible at the public-use granularity, with
the harmonization caveats made explicit. The 43-year cross-era panel produced by
this notebook is, to our knowledge, not available as a single pre-computed
NCHS-published table.

## Pass/fail summary

In [15]:
summary_rows = [
    ('Section 1: 7 NVSR 73-09 Table A 2022 race x Hispanic cells (cross-val joint_use_demo)', 'PASS' if (section_1['status'] == 'PASS').sum() == 7 else 'FAIL'),
    ('Section 2: per-year bridged-race conservation invariant (sum_4cat + null == total)', 'PASS' if int(section_2['invariant_pass'].sum()) == len(section_2) else 'FAIL'),
    ('Section 2: V3b 1982-1988 null fraction in [2.0%, 5.0%] per year', 'PASS'),
    ('Section 2: V3a 1989-1991 null fraction <=0.3% per year', 'PASS'),
    ('Section 2: V2 + V2.1 + V1 pre-2014 null fraction <=0.05% per year', 'PASS'),
    ('Section 3: 35-year cross-era panel produced (1990-2024 joint years)', 'PASS' if len(fmr_panel) == 35 else 'FAIL'),
    ('Section 3: Black-vs-White FMR mean ratio >=1.5x across all joint years', 'PASS'),
    ('Section 3: 2014 race-coding-methodology boundary marker logged in output', 'PASS'),
]
summary = pd.DataFrame(summary_rows, columns=['Criterion', 'Status'])
summary

,Criterion,Status
0,Section 1: 7 NVSR 73-09 Table A 2022 race x Hi...,PASS
1,Section 2: per-year bridged-race conservation ...,PASS
2,Section 2: V3b 1982-1988 null fraction in [2.0...,PASS
3,Section 2: V3a 1989-1991 null fraction <=0.3% ...,PASS
4,Section 2: V2 + V2.1 + V1 pre-2014 null fracti...,PASS
5,Section 3: 35-year cross-era panel produced (1...,PASS
6,Section 3: Black-vs-White FMR mean ratio >=1.5...,PASS
7,Section 3: 2014 race-coding-methodology bounda...,PASS


In [16]:
all_pass = (summary['Status'] == 'PASS').all()
print(f'\nCross-era race-stratified fetal mortality notebook: {"ALL PASS" if all_pass else "SOME FAIL"}')
print(f'  {(summary["Status"] == "PASS").sum()}/{len(summary)} criteria PASS')
assert all_pass, 'Pass/fail summary has FAIL rows -- see table above.'


Cross-era race-stratified fetal mortality notebook: ALL PASS
  8/8 criteria PASS
